## 🤖 Building MediAgent: A Medical Q&A Agent with Gemma-3n and LangChain

*Authored by: [Hyunji Jeon](https://github.com/HyunZ118)*

Hello! In this recipe, we'll build "MediAgent," a specialized AI agent for the medical field, utilizing Google's Gemma-3n model and LangChain. 

MediAgent helps patients easily understand complex and technical discharge summaries, ensuring they don't miss crucial information.

To effectively respond to the various types of questions a patient might have, the architecture is designed to use an LLM as a Router. 

It first identifies the user's intent and then dynamically selects the most appropriate of three specialized tools to generate an answer:

1. Term Explainer: Simplifies difficult medical terms like "PO" or "BID," explaining them from a patient's perspective.

2. Instruction Extractor: Accurately extracts specific information from the document to answer questions like, "When is my next appointment?" or "What medicine should I take?"

3. General Q&A (RAG): Handles general questions about the patient's overall condition or hospitalization, which require referencing the entire document for a comprehensive answer.

Now, let's start building MediAgent! 🚀

### 1. Setting Up Our Environment 

First, let's install the necessary libraries and import all the modules we'll need for this project. 

We'll also define the key configuration variables, such as the model IDs we'll be using.

In [ ]:
# Step 1: Install dependencies
#!pip install -q transformers torch accelerate bitsandbytes langchain langchain_community sentence-transformers faiss-gpu

# Step 2: Import necessary modules
import torch
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import LLMChain, RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
import re

# Step 3: Define key configurations
MODEL_ID = "google/gemma-3n-e4b-it" 
DOCUMENT_PATH = "discharge_summary_en.txt"
EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

### 2. Loading the Core LLM (Gemma-3n)

Now, we'll load our primary Large Language Model, Gemma-3n. 

This model will be the engine powering all our agent's tools and its "brain." 

We use the HuggingFacePipeline from LangChain to easily integrate it.

In [ ]:
print("Loading the Gemma-3n model... (This may take a moment)")
llm = HuggingFacePipeline.from_model_id(
    model_id=MODEL_ID,
    task="text-generation",
    device_map="auto",
    model_kwargs={"torch_dtype": "auto"},
    pipeline_kwargs={"max_new_tokens": 512, "return_full_text": False, "temperature": 0.1},
)
print("✅ Model loaded successfully!")

### 3. Creating the Sample Discharge Summary

Our agent needs a source of truth to answer questions accurately. 

For this recipe, we'll create a sample patient discharge summary and save it as a text file. 

This document will serve as the agent's knowledge base.

In [ ]:
discharge_summary_text = """
Discharge Summary

Patient Name: John Doe
Date of Discharge: 2025-10-27

1. Diagnosis:
Patient was admitted for acute appendicitis and underwent a successful appendectomy.

2. Post-Procedure Care & Instructions:
- Medications: Take 1 tablet of Acetaminophen PO BID for pain management.
- Wound Care: Keep the incision area clean and dry. Avoid soaking in a bath for 2 weeks.
- Activity: No heavy lifting for 4 weeks. Light walking is encouraged.
- Diet: Start with a liquid diet and slowly progress to solid foods. Avoid spicy food.
- Follow-up: A follow-up appointment is scheduled with Dr. Smith on 2025-11-10.

3. Signs to Watch For:
If you experience high fever, severe pain, or redness around the incision, contact the hospital immediately. NPO status is no longer required.
"""
with open(DOCUMENT_PATH, "w", encoding="utf-8") as f:
    f.write(discharge_summary_text)

print(f"✅ Discharge summary created at '{DOCUMENT_PATH}'")

### 4. Building the Agent's Toolbox

An agent is only as good as its tools. We will create three specialized tools to handle different types of patient queries.

#### 4.1. ⚙️ Tool 1: The Term Explainer

This tool is for explaining complex medical jargon in simple terms.

In [4]:
def create_term_explainer_tool(llm_model):
    """Creates a tool to explain medical terms."""
    prompt = PromptTemplate(
        input_variables=["term"],
        template="""<start_of_turn>user
You are a friendly pharmacist. Explain the medical term "{term}" in a very simple way for a patient.
Your Explanation:
<end_of_turn><start_of_turn>model"""
    )
    return LLMChain(llm=llm_model, prompt=prompt)

#### 4.2. ⚙️ Tool 2: The Instruction Extractor

This tool is designed to extract specific instructions (like appointment dates or medication details) directly from the text.

In [5]:
def create_instruction_extractor_tool(llm_model, document_path):
    """Creates a tool to extract specific information from the document."""
    with open(document_path, 'r', encoding='utf-8') as f:
        doc_text = f.read()
    
    template_string = """<start_of_turn>user
You are a helpful assistant. From the document below, extract the specific information related to the patient's "{topic}".
If you can't find it, say "I could not find information on that topic in the document."

**Document:**
{doc_text}

**Extracted Information about "{topic}":**
<end_of_turn><start_of_turn>model"""
    
    prompt = PromptTemplate(
        input_variables=["topic", "doc_text"],
        template=template_string
    ).partial(doc_text=doc_text)
    
    return LLMChain(llm=llm_model, prompt=prompt)

#### 4.3. ⚙️ Tool 3: The General Q&A Tool (RAG)

For any other general questions, we'll use a Retrieval-Augmented Generation (RAG) tool. It finds the most relevant parts of the document and uses them as context to answer the question.

In [6]:
def create_general_qa_tool(llm_model, document_path):
    """Creates a RAG tool for general questions."""
    loader = TextLoader(document_path, encoding='utf-8')
    docs = loader.load_and_split(CharacterTextSplitter.from_tiktoken_encoder(chunk_size=200, chunk_overlap=30))
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_ID)
    vector_db = FAISS.from_documents(docs, embeddings)
    
    qa_prompt = PromptTemplate(
        template="""<start_of_turn>user
Based ONLY on the provided context, answer the question clearly.
Context: {context}
Question: {question}
Answer:<end_of_turn><start_of_turn>model""",
        input_variables=["context", "question"])
        
    return RetrievalQA.from_chain_type(
        llm=llm_model,
        chain_type="stuff",
        retriever=vector_db.as_retriever(),
        chain_type_kwargs={"prompt": qa_prompt})

### 5. Creating the Agent's Brain: The Router 🤔

This is the most critical part of our agent. The router is an LLMChain whose sole job is to analyze the user's query and classify it into one of our tool categories (explainer, extractor, or qa). This decision will determine which tool gets used.

In [7]:
def create_router_chain(llm_model):
    """Creates the router chain to classify the user's query."""
    prompt = PromptTemplate(
        input_variables=["query"],
        template="""<start_of_turn>user
Your job is to classify the user's query into one of three categories: 'explainer', 'extractor', or 'qa'.
- 'explainer': Use for questions asking the meaning of a specific term (e.g., "What is PO?").
- 'extractor': Use for questions asking about specific instructions (e.g., "When is my next appointment?", "What medicine should I take?").
- 'qa': Use for all other general questions about the patient's condition or hospital stay.
- 'chitchat': Use for conversational fillers, greetings, or expressions of gratitude like "thank you", "okay", "got it", "sounds good".

Respond with only the category name and nothing else.

Query: "{query}"
Category:
<end_of_turn><start_of_turn>model"""
    )
    return LLMChain(llm=llm_model, prompt=prompt)

### 6. Running MediAgent 🤖

Finally, let's bring everything together! 

We will instantiate our tools and the router, and then create a loop to interact with our agent.


The flow is simple:

1) Get the patient's question.

2) Use the Router to determine the intent.

3) Execute the appropriate Tool based on the intent.

4) Display the answer.

In [ ]:
# --- Instantiate all tools and the router ---
explainer_tool = create_term_explainer_tool(llm)
extractor_tool = create_instruction_extractor_tool(llm, DOCUMENT_PATH)
qa_tool = create_general_qa_tool(llm, DOCUMENT_PATH)
router = create_router_chain(llm)

print("\n\n----- Starting the 🤖 MediAgent: Discharge Summary Agent -----")
print("\nHello! Ask me anything about your discharge summary.")
print("Examples: 'What does PO mean?', 'When is my next appointment?', 'What was my diagnosis?'")

# --- Main interaction loop ---
while True:
    user_query = input("\n[Patient 🤕] >> ")
    if user_query.lower().strip() in ["exit", "quit"]:
        print("\n[MediAgent 🤖] >> Thank you for using the agent. Have a great day!")
        break
    
    elif not user_query.strip():
        print("\n[MediAgent 🤖] >> How can I help? Please ask a question based on the summary.")
        print("                   Examples: 'What does PO mean?', 'When is my next appointment?'")
        continue

    # 1. Use the router to determine intent
    raw_intent_text = router.invoke({"query": user_query})['text'].lower()
    
    intent = ""
    if "explainer" in raw_intent_text:
        intent = "explainer"
    elif "extractor" in raw_intent_text:
        intent = "extractor"
    elif "chitchat" in raw_intent_text:
        intent = "chitchat"
    else: 
        intent = "qa"
    
    print("\n\n--------------------------------------------------------------")
    print(f"[Patient Ask 🤕] >> {user_query}")
    print(f"               (🤔Router decided on -> ⚙️Tool: {intent})") # Optional: for debugging

    # 2. Execute the appropriate tool
    final_answer = ""
    
    if intent == "chitchat":
        print("\n\n[MediAgent 🤖] >> Of course. Do you have any more questions? If you'd like to end the conversation, please type 'exit'.")
        continue

    if intent == "explainer":
        match = re.search(r'(?:is|mean|about)\s+(.+)\??', user_query, re.IGNORECASE)
        if match:
            term = match.group(1).strip()
            final_answer = explainer_tool.invoke({"term": term})['text']
        else:
            final_answer = "Could you please rephrase that? For example, say 'What is [term]?'"
    
    elif intent == "extractor":
        final_answer = extractor_tool.invoke({"topic": user_query})['text']
        
    elif intent == "qa":
        final_answer = qa_tool.invoke({"query": user_query})['result']
        
    print(f"\n\n[MediAgent 🤖] >> {final_answer.strip()}")

/tmp/ipykernel_1654469/19825536.py:10: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  return LLMChain(llm=llm_model, prompt=prompt)
/tmp/ipykernel_1654469/3438452672.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_ID)




----- Starting the 🤖 MediAgent: Discharge Summary Agent -----

Hello! Ask me anything about your discharge summary.
Examples: 'What does PO mean?', 'When is my next appointment?', 'What was my diagnosis?'


--------------------------------------------------------------
[Patient Ask 🤕] >> what is PO?
               (🤔Router decided on -> ⚙️Tool: explainer)


[MediAgent 🤖] >> Hi there! 😊

You asked about "PO," and it's a very common abbreviation you'll see on prescriptions. It simply stands for **"by mouth."** 

So, if a prescription says "Take 2 tablets PO," it means you should just swallow the tablets with water. Easy peasy! 

Think of it like this: PO just tells you *how* to take the medicine – straight into your mouth! 

Do you have any other questions about your medication or anything else I can help you with today?


--------------------------------------------------------------
[Patient Ask 🤕] >> when is my next appointment?
               (🤔Router decided on -> ⚙️Tool: extracto

### 7. Conclusion & Next steps

Congratulations! You've successfully built MediAgent, a smart medical assistant powered by Gemma-3n and LangChain.

Did you find this interesting? Want to explore more with the Gemma-3n model?
Here are a few ideas to take this project even further:

- Build New Tools : You could develop new tools, such as one for scheduling follow-up appointments.
- Integrate Memory: You can incorporate conversational memory so the agent can remember previous parts of the conversation.
- Deploy as a Service: You can build MediAgent into a user-friendly web interface using tools like Gradio.

We hope this recipe inspires you to build your own amazing agent-based applications! 🤗